In [ ]:
import scanpy as sc
import cell2location
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import logging
from tqdm import tqdm
import warnings

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# Configure warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

class SpatialAnalysis:
    def __init__(self, visium_path_infected, visium_path_uninfected, sc_adata, output_dir="./results"):
        """Initialize spatial analysis"""
        logger.info("Loading data...")
    
        # Load Visium data
        try:
            self.visium_infected = sc.read_visium(visium_path_infected)
            self.visium_uninfected = sc.read_visium(visium_path_uninfected)
            
            # Make gene names unique immediately
            self.visium_infected.var_names_make_unique()
            self.visium_uninfected.var_names_make_unique()
            
        except Exception as e:
            logger.error(f"Failed to load Visium data: {str(e)}")
            raise
    
        # Store and process single-cell data
        self.sc_adata = sc_adata.copy()
        self.sc_adata.var_names_make_unique()
    
        # Setup output directory
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        # Setup data
        self._setup_data()
        
    def _setup_data(self):
        """Prepare and validate data for analysis"""
        logger.info("Setting up data...")
        
        try:
            # Verify spatial coordinates
            for adata, label in [(self.visium_infected, "infected"), 
                                (self.visium_uninfected, "uninfected")]:
                if 'spatial' not in adata.obsm:
                    raise ValueError(f"No spatial coordinates found in {label} dataset")
            
            # Ensure raw counts are used
            if 'raw_counts' in self.sc_adata.layers:
                self.sc_adata.X = self.sc_adata.layers['raw_counts'].copy()
            
            # Get common genes
            common_genes = list(set(self.visium_infected.var_names) &
                                set(self.visium_uninfected.var_names) &
                                set(self.sc_adata.var_names))
            
            if len(common_genes) == 0:
                raise ValueError("No common genes found between datasets")
            
            logger.info(f"Found {len(common_genes)} common genes across datasets")
            
            # Subset to common genes
            self.visium_infected = self.visium_infected[:, common_genes].copy()
            self.visium_uninfected = self.visium_uninfected[:, common_genes].copy()
            self.sc_adata = self.sc_adata[:, common_genes].copy()
            
        except Exception as e:
            logger.error(f"Data setup failed: {str(e)}")
            raise

    def run_deconvolution(self, n_epochs=1000, batch_size=2500, gpu=True):
        """Run cell2location deconvolution with condition-specific reference"""
        logger.info("Running cell type deconvolution...")
        
        try:
            if gpu and not torch.cuda.is_available():
                logger.warning("GPU requested but not available. Falling back to CPU.")
                gpu = False
            
            accelerator = "gpu" if gpu else "cpu"
            
            # Split reference data by condition
            uninfected_sc = self.sc_adata[self.sc_adata.obs['timepoint'] == '0wk'].copy()
            infected_sc = self.sc_adata[self.sc_adata.obs['timepoint'] == '3wk'].copy()
            
            logger.info(f"Reference data split: {len(uninfected_sc)} uninfected cells, {len(infected_sc)} infected cells")
            
            # Process each condition separately
            for condition, visium_data, ref_data in [
                ('infected', self.visium_infected, infected_sc),
                ('uninfected', self.visium_uninfected, uninfected_sc)
            ]:
                logger.info(f"Processing {condition} condition...")
                
                # Ensure batch information is properly set up
                if 'dataset' in ref_data.obs.columns:
                    ref_data.obs['batch'] = ref_data.obs['dataset'].astype('category')
                else:
                    logger.error("No 'dataset' column found in reference data")
                    raise ValueError("Reference data must contain a 'dataset' column")
                
                logger.info("Setting up reference model...")
                
                # Setup reference model for this condition
                cell2location.models.RegressionModel.setup_anndata(
                    ref_data,
                    batch_key='batch',
                    labels_key='celltypes',  # Make sure this matches your column name
                    categorical_covariate_keys=['experiment']
                )
                
                ref_model = cell2location.models.RegressionModel(ref_data)
                ref_model.train(
                    max_epochs=n_epochs,
                    accelerator=accelerator,
                    progress_bar_refresh_rate=1
                )
                
                # Export posterior
                ref_data = ref_model.export_posterior(
                    ref_data,
                    sample_kwargs={'num_samples': 1000, 'batch_size': batch_size}
                )
                
                # Extract cluster-specific expression
                if 'means_per_cluster_mu_fg' in ref_data.varm.keys():
                    inf_aver = ref_data.varm['means_per_cluster_mu_fg'][[
                        f'means_per_cluster_mu_fg_{i}' for i in ref_data.uns['mod']['factor_names']
                    ]].copy()
                else:
                    inf_aver = ref_data.var[[
                        f'means_per_cluster_mu_fg_{i}' for i in ref_data.uns['mod']['factor_names']
                    ]].copy()
                
                inf_aver.columns = ref_data.uns['mod']['factor_names']
                ref_summary = inf_aver
                
                # Setup and run cell2location model
                cell2location.models.Cell2location.setup_anndata(visium_data)
                
                model = cell2location.models.Cell2location(
                    visium_data,
                    cell_state_df=ref_summary,
                    N_cells_per_location=30,
                    detection_alpha=20
                )
                
                model.train(
                    max_epochs=n_epochs,
                    accelerator=accelerator,
                    progress_bar_refresh_rate=1
                )
                
                # Export results
                visium_data = model.export_posterior(
                    visium_data,
                    sample_kwargs={'num_samples': 1000, 'batch_size': model.adata.n_obs}
                )
                
                # Store results
                if condition == 'infected':
                    self.visium_infected = visium_data
                else:
                    self.visium_uninfected = visium_data
                    
            return True
            
        except Exception as e:
            logger.error(f"Pipeline failed: {str(e)}")
            if gpu:
                torch.cuda.empty_cache()
            raise
            
    def visualize_celltypes(self, output_dir="./celltype_plots", max_cells_per_plot=1):
        """Visualize spatial distribution of cell types using cell2location plotting"""
        logger.info("Visualizing cell type spatial distributions...")
    
        try:
            from cell2location.plt import plot_spatial
            output_dir = Path(output_dir)
            output_dir.mkdir(exist_ok=True, parents=True)
        
            for condition, adata in [
                ('Infected', self.visium_infected),
                ('Uninfected', self.visium_uninfected)
            ]:
                logger.info(f"Processing {condition} dataset...")
            
                # Get cell type names from the model
                if 'mod' not in adata.uns:
                    logger.error(f"No model results found in {condition} dataset")
                    continue
                
                cell_types = adata.uns['mod']['factor_names']
                logger.info(f"Found {len(cell_types)} cell types")
            
                # Add cell abundance data to obs
                adata.obs[cell_types] = adata.obsm['q05_cell_abundance_w_sf']
            
                # Process cell types in groups of max_cells_per_plot
                for i in range(0, len(cell_types), max_cells_per_plot):
                    group = cell_types[i:i + max_cells_per_plot]
                
                    try:
                        with plt.rc_context({'figure.figsize': (15, 15)}):
                            fig = plot_spatial(
                                adata=adata,
                                color=group,  # cell types to plot
                                labels=group,  # labels to show
                                show_img=True,
                                style='fast',
                                max_color_quantile=0.992,
                                circle_diameter=6,
                                colorbar_position='right'
                            )
                        
                            # Save figure
                            group_name = f"celltypes_{i//max_cells_per_plot}"
                            output_file = output_dir / f"{condition}_{group_name}.png"
                            plt.savefig(
                                output_file,
                                dpi=300,
                                bbox_inches='tight',
                                facecolor='white'
                            )
                            plt.close()
                        
                    except Exception as e:
                        logger.error(f"Error plotting group {i//max_cells_per_plot} in {condition}: {str(e)}")
                        plt.close()
                        continue
            
                # Clean up added obs columns
                adata.obs = adata.obs.drop(columns=cell_types)
                    
            return True
        
        except Exception as e:
            logger.error(f"Cell type visualization failed: {str(e)}")
            plt.close('all')
            raise

# Example usage:
def main():
    # Set paths
    paths = {
        'visium_infected': '/home/robeylab/Desktop/Spleen_Visium_Exp3A_V1S1_3wk_infected/outs',
        'visium_uninfected': '/home/robeylab/Desktop/outs_UNINFECTED',
        'output': './results_split_training'
    }
    
    # Load single-cell data
    sc_data = sc.read_h5ad('/home/robeylab/cellxgene_data/preprocessed_deconvolution_sc_spleen_240116_updated.h5ad')
    
    # Initialize analyzer
    analyzer = SpatialAnalysis(
        visium_path_infected=paths['visium_infected'],
        visium_path_uninfected=paths['visium_uninfected'],
        sc_adata=sc_data,
        output_dir=paths['output']
    )
    
    # Run deconvolution
    analyzer.run_deconvolution(n_epochs=1000, batch_size=2500, gpu=True)
    
    # Visualize results
    analyzer.visualize_celltypes(output_dir=paths['output'])

if __name__ == "__main__":
    main()